In [1]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

# Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [2]:
import pandas as pd

data = pd.read_csv("Data.csv")
data.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

A Chain is the execution engine that makes the template actually do something.

Think of a LangChain Chain similarly to structuring a deep learning pipeline, like a PyTorch nn.Sequential container.

You have different pieces, and the Chain is the wrapper that binds them together and pushes the data through them in the correct order.

In [3]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

llm = ChatOpenAI(model = "gpt-3.5-turbo", temperature = 0.9)

In [4]:
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

In [5]:
chain = LLMChain(llm = llm, prompt = prompt)
chain

LLMChain(memory=None, callbacks=None, callback_manager=None, verbose=False, prompt=ChatPromptTemplate(input_variables=['product'], output_parser=None, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['product'], output_parser=None, partial_variables={}, template='What is the best name to describe     a company that makes {product}?', template_format='f-string', validate_template=True), additional_kwargs={})]), llm=ChatOpenAI(verbose=False, callbacks=None, callback_manager=None, client=<class 'openai.api_resources.chat_completion.ChatCompletion'>, model_name='gpt-3.5-turbo', temperature=0.9, model_kwargs={}, openai_api_key=None, openai_api_base=None, openai_organization=None, request_timeout=None, max_retries=6, streaming=False, n=1, max_tokens=None), output_key='text')

In [6]:
product = "Queen Size Sheet Set"
chain.run(product)

'Regal Linens Co.'

When you call `chain.run()`, the Chain handles these steps automatically:

1. Takes your raw input variables
2. Injects them into the Prompt Template to format the final string
3. Sends that formatted string to the LLM
4. Receives the generated text from the LLM and returns it to you


## SimpleSequentialChain

This is designed for a very specific scenario: chaining multiple tasks together where the output of Step 1 is exactly what Step 2 needs as its input.

In [7]:
from langchain.chains import SimpleSequentialChain

In [8]:
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

chain_one = LLMChain(llm = llm, prompt = first_prompt)

In [9]:
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following \
    company:{company_name}"
)

chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [10]:
overall_simple_chain = SimpleSequentialChain(chains = [chain_one, chain_two], verbose = True)

In [11]:
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
Regal Linens Co.
Regal Linens Co. offers luxurious and high-quality linens for bedrooms and bathrooms, providing elegance and comfort for your home.

> Finished chain.


'Regal Linens Co. offers luxurious and high-quality linens for bedrooms and bathrooms, providing elegance and comfort for your home.'

So it's:

`product -> input to chain 1 -> output of chain 1 -> input to chain 2 -> final output`


## SequentialChain

| Feature | SimpleSequentialChain | SequentialChain |
|---|---|---|
| Inputs allowed | Only 1 | Multiple |
| Outputs allowed | Only 1 (the final chain's output) | Multiple (can return outputs from intermediate chains) |
| Data flow | Strict A → B → C | Flexible (Chain C can use outputs from Chain A and the original user input) |
| Variable tracking | Handled automatically (blind handoff) | Requires explicit naming of `input_variables` and `output_variables` |


In [12]:
from langchain.chains import SequentialChain

In [15]:
# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="English_Review"
                    )


In [16]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary"
                    )


In [17]:
# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )


In [18]:

# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )


In [19]:
overall_chain = SequentialChain(
    chains = [chain_one, chain_two, chain_three, chain_four], #order of the chain in which it is executed
    input_variables = ["Review"],
    output_variables = ["English_Review", "summary", "followup_message"],
    verbose = True
)

In [21]:
overall_chain(data.Review[5])



> Entering new SequentialChain chain...

> Finished chain.


{'Review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'English_Review': '"I find the taste mediocre. The foam doesn\'t last, it\'s strange. I buy the same ones in stores and the taste is much better... Old batch or counterfeit!?"',
 'summary': 'The reviewer found the taste of the product to be mediocre, with the foam not lasting, leading them to suspect the product may be an old batch or counterfeit.',
 'followup_message': "Merci pour votre avis honnête sur le produit. Il est regrettable d'apprendre que la saveur n'était que moyenne et que la mousse ne durait pas. Votre soupçon selon lequel il pourrait s'agir d'un lot ancien ou contrefait est compréhensible. Il est important de toujours acheter des produits authentiques pour garantir une expérience de qualité. Peut-être pourriez-vous contacter le fabricant pour obtenir plus d'informations sur la provenance du pro

## Shared dict concept

Unlike `SimpleSequentialChain` (which just blindly hands the output of A to B), `SequentialChain` manages a growing dictionary of variables. Every time a chain finishes, it drops its output into this shared dictionary. When the next chain starts, it looks inside that dictionary for the specific variables it needs.


## Router chain

If standard sequential chains are like a straight assembly line, a Router Chain is like a call center switchboard: it reads the user's input and dynamically decides which specialized "expert" chain should handle the request.

In [24]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [25]:
#defining all the prompt templates
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

In [26]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

In [33]:
#first actually create those experts

destination_chains = {} #stores the chain objects 

for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template = prompt_template) 
    chain = LLMChain(llm = llm, prompt = prompt)
    destination_chains[name] = chain
    
#builds a "name: description" string for each expert - this is what gets fed into the router prompt so the LLM knows which experts are available and what each one is good for
destinations = [f"{p_info['name']}: {p_info['description']}" for p_info in prompt_infos]
destination_str = "\n".join(destinations)
destination_str


'physics: Good for answering questions about physics\nmath: Good for answering math questions\nHistory: Good for answering history questions\ncomputer science: Good for answering computer science questions'

In [35]:
#default chain to call when the input matches nothing specified above

default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm = llm, prompt = default_prompt)

In [37]:
#this is the main prompt which tells to the LLM what to do exactly
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: The value of “destination” MUST match one of \
the candidate prompts listed below.\
If “destination” does not fit any of the specified prompts, set it to “DEFAULT.”
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [39]:
#setting the menu for the LLM
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations = destination_str,
) #formatting the prompt

router_prompt = PromptTemplate(
    template = router_template,
    input_variables = ["input"],
    output_parser = RouterOutputParser(),
) #creating the prompt template object, with a parser attached so the router's output gets turned into a usable dict

router_chain = LLMRouterChain.from_llm(llm, router_prompt) #creating the router chain itself


The MultiPromptChain is the master controller. This specific code block represents the Routing architecture.

In [41]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [42]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation is the thermal electromagnetic radiation emitted by a perfect black body, which is an idealized physical body that absorbs all incident electromagnetic radiation. This type of radiation follows specific laws, known as Planck's law of black body radiation, which describes the intensity and distribution of radiation emitted by a black body at different wavelengths and temperatures. In simpler terms, it is the radiation given off by an object that absorbs all incoming light and energy."

In [43]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
None: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA is the genetic material that carries the instructions for building and maintaining the structure and function of our cells and bodies. DNA contains the information necessary for the synthesis of proteins, which are essential for all cellular processes. Without DNA, cells would not be able to replicate and carry out their specialized functions, ultimately leading to the breakdown of the organism. Therefore, DNA is crucial for the survival and function of every cell in our body.'